# minibayes.viz Showcase

This notebook demonstrates all visualization functions in `minibayes.viz` using a simple multi-chain linear regression example.

In [ ]:
import numpy as np
import minibayes as mb
from minibayes import viz

# Set random seed for reproducibility
np.random.seed(42)

## 1. Generate Synthetic Data

Simple linear regression: $y = \alpha + \beta x + \epsilon$ where $\epsilon \sim N(0, \sigma^2)$

In [ ]:
# True parameters
true_alpha = 1.5
true_beta = 2.0
true_sigma = 0.5

# Generate data
n_data = 30
x = np.linspace(0, 3, n_data)
y = true_alpha + true_beta * x + np.random.normal(0, true_sigma, n_data)

print(f"Data: {n_data} points")
print(f"True parameters: alpha={true_alpha}, beta={true_beta}, sigma={true_sigma}")

## 2. Define and Fit the Model

In [ ]:
# Define priors
def priors(p):
    p("alpha", mb.dist.Normal(0, 10))
    p("beta", mb.dist.Normal(0, 10))
    p("sigma", mb.dist.HalfNormal(3))


# Define log-likelihood
def log_likelihood(params, data=None):
    mu = params["alpha"] + params["beta"] * x
    return mb.dist.Normal(mu, params["sigma"]).obs_logp(y)

# Create model
model = mb.Model(priors=priors, log_likelihood=log_likelihood)

print(f"Model parameters: {model.param_names}")

In [ ]:
# Run MCMC with multiple chains
result = mb.sample(
    model,
    num_samples=5000,
    num_warmup=2000,
    num_chains=4,
    sampler="adaptive_mh",
    seed=42,
    progress=True,
)

print(f"\nSampling complete!")
print(f"Chains: {result.num_chains}, Samples per chain: {result.num_samples}")
print(f"Acceptance rates: {result.acceptance_rate}")

---

## 3. Visualization Showcase

### 3.1 `summary_table()` - Formatted Summary Statistics

In [ ]:
print(viz.summary_table(result))

### 3.2 `plot_density()` - Posterior Histograms

In [ ]:
fig = viz.plot_density(result)

# Add true values as reference
import matplotlib.pyplot as plt
axes = fig.get_axes()
true_vals = {"alpha": true_alpha, "beta": true_beta, "sigma": true_sigma}
for ax in axes:
    param = ax.get_title().replace("Posterior: ", "")
    if param in true_vals:
        ax.axvline(true_vals[param], color="#85C1A3", linestyle=":", linewidth=2, label=f"true={true_vals[param]}")
        ax.legend(fontsize=8)
plt.show()

### 3.3 `plot_samples()` - Samples Over Iteration (Convergence Check)

In [ ]:
fig = viz.plot_samples(result)
plt.show()

### 3.4 `plot_autocorr()` - Autocorrelation by Lag

In [ ]:
fig = viz.plot_autocorr(result, max_lag=40)
plt.show()

### 3.5 `plot_forest()` - Parameter Estimates with Credible Intervals

In [ ]:
# Box plots showing parameter distributions
fig = viz.plot_forest(result)

plt.show()

### 3.6 `plot_predictive()` - Posterior Predictive with Uncertainty Bands

In [ ]:
# Generate posterior predictive samples
x_plot = np.linspace(-0.5, 4, 100)

def predictive_fn(params, rng):
    mu = params["alpha"] + params["beta"] * x_plot
    return {"y_pred": mb.dist.Normal(mu, params["sigma"]).sample(size=len(x_plot), rng=rng)}

ppc = result.predict(predictive_fn, num_samples=500, seed=42)
y_pred = ppc["y_pred"]

print(f"Posterior predictive shape: {y_pred.shape}")

In [ ]:
# Plot posterior predictive with observed data
fig = viz.plot_predictive(x_plot, y_pred, x_obs=x, y_obs=y, credible_interval=0.9)

plt.show()

---

## 4. Composable Plots

All viz functions support custom axes for composing multi-panel figures.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# Plot density for alpha
viz.plot_density(result, params=["alpha"], ax=axes[0, 0])
axes[0, 0].axvline(true_alpha, color="#85C1A3", linestyle=":", linewidth=2)

# Plot samples for alpha
viz.plot_samples(result, params=["alpha"], ax=axes[0, 1])

# Plot density for beta
viz.plot_density(result, params=["beta"], ax=axes[1, 0])
axes[1, 0].axvline(true_beta, color="#85C1A3", linestyle=":", linewidth=2)

# Plot samples for beta
viz.plot_samples(result, params=["beta"], ax=axes[1, 1])

plt.suptitle("Custom Multi-Panel Layout", fontsize=12, color="#4A4A4A")
plt.tight_layout()
plt.show()

---

## 5. Style Context Manager

Use `viz.style()` to apply the minibayes pastel style to any matplotlib plot.

In [ ]:
from minibayes.viz import style, PALETTE, CHAIN_COLORS

print("Palette colors:")
for name, color in PALETTE.items():
    print(f"  {name}: {color}")

print(f"\nChain colors: {CHAIN_COLORS}")

In [ ]:
# Using the style context manager for custom plots
with style():
    fig, ax = plt.subplots(figsize=(6, 4))

    # Custom scatter plot with pastel colors
    ax.scatter(x, y, c=PALETTE["terracotta"], s=50, alpha=0.8, label="Data")
    ax.plot(x, true_alpha + true_beta * x, c=PALETTE["sage"], linewidth=2, label="True")

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Custom Plot with minibayes Style")
    ax.legend()

    plt.show()

---

## Summary

| Function | Purpose |
|----------|--------|
| `viz.plot_density()` | Posterior histograms |
| `viz.plot_samples()` | Samples over iteration |
| `viz.plot_autocorr()` | Autocorrelation by lag |
| `viz.plot_forest()` | Parameter estimates with CI |
| `viz.plot_predictive()` | Predictions with uncertainty |
| `viz.summary_table()` | Formatted summary statistics |

All functions accept `InferenceResult`, dict of samples, or raw arrays for flexibility.